In [2]:
import torch

In [3]:
def f(x):
    x1 = x[0]
    x2 = x[1]
    
    # f(x₁, x₂) = 3x₁² + 5eˣ²
    return 3 * x1**2 + 5 * torch.exp(x2)

def analytical_gradient(x):
    partial_x1 = 6 * x[0]
    partial_x2 = 5 * torch.exp(x[1])
    
    # 각 변수에 대한 편미분을 하나의 gradient 벡터로
    return torch.stack((partial_x1, partial_x2))

In [4]:
# gradient를 계산할 위치 x=(2, 0)을 지정
# 입력 shape는 (2,), 함수 출력 shape는 ()
x = torch.tensor([2.0, 0.0], dtype=torch.float64)

function_value = f(x)
expected_gradient = analytical_gradient(x)

print("x:", x)
print("f(x):", function_value)
print("Analytical gradient:", expected_gradient)
print("Gradient shape:", expected_gradient.shape)

x: tensor([2., 0.], dtype=torch.float64)
f(x): tensor(17., dtype=torch.float64)
Analytical gradient: tensor([12.,  5.], dtype=torch.float64)
Gradient shape: torch.Size([2])


In [5]:
# 각 좌표축 방향으로 아주 조금 이동시켜 편미분을 수치적으로 근사한다.
# 중앙 차분 공식: [f(x+h)-f(x-h)] / (2h)
h = 1e-5
numerical_gradient = torch.empty_like(x)

for index in range(x.numel()):
    # initialize: 현재 index의 변수만 h만큼 변화시키고 나머지는 고정
    perturbation = torch.zeros_like(x)
    perturbation[index] = h

    numerical_gradient[index] = (
        f(x + perturbation) - f(x - perturbation)) / (2 * h)

print("Analytical gradient:", expected_gradient)
print("Numerical gradient:", numerical_gradient)

assert torch.allclose(
    numerical_gradient,
    expected_gradient,
    atol=1e-5,
)

Analytical gradient: tensor([12.,  5.], dtype=torch.float64)
Numerical gradient: tensor([12.0000,  5.0000], dtype=torch.float64)


In [6]:
# 이차형식 q(x)=xᵀAx의 gradient 공식을 확인한다.
A = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0],
])

x = torch.tensor([1.0, 2.0])

# x @ A @ x의 출력은 0차원 scalar tensor
quadratic_value = x @ A @ x

# A가 대칭행렬이 아닐 수도 있으므로 A와 A.T가 모두 필요하다.
quadratic_gradient = (A + A.T) @ x

print("Quadratic value:", quadratic_value)
print("Gradient:", quadratic_gradient)
print("Gradient shape:", quadratic_gradient.shape)


Quadratic value: tensor(27.)
Gradient: tensor([12., 21.])
Gradient shape: torch.Size([2])


In [7]:
# 행렬의 squared Frobenius norm은 모든 원소 제곱의 합
X = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0],
])

squared_frobenius_norm = X.pow(2).sum()

# 각 원소 X[i, j]²을 미분하면 2X[i, j]가 된다.
frobenius_gradient = 2 * X

print("Squared Frobenius norm:", squared_frobenius_norm)
print("Gradient of X:")
print(frobenius_gradient)

assert squared_frobenius_norm.item() == 30.0
assert frobenius_gradient.shape == X.shape
assert torch.equal(frobenius_gradient, 2 * X)

Squared Frobenius norm: tensor(30.)
Gradient of X:
tensor([[2., 4.],
        [6., 8.]])
